In [16]:
import matplotlib.pyplot as plt
from moku.instruments import MultiInstrument, WaveformGenerator, LockInAmp, DigitalFilterBox, ArbitraryWaveformGenerator
import time
import os
import numpy as np
import traceback

m = MultiInstrument('192.168.2.XX', force_connect=True,platform_id=4)

# Set instrument slots in Multi-Instrument Mode
awg = m.set_instrument(slot=1,instrument=ArbitraryWaveformGenerator)
wg = m.set_instrument(slot=2,instrument=WaveformGenerator)
lia = m.set_instrument(slot=3,instrument=LockInAmp)
dfb = m.set_instrument(slot=4,instrument=DigitalFilterBox)

# Set Multi-Instrument Mode connections
connections=[dict(source='Slot1OutA', destination='Slot2InA'),
                dict(source='Slot2OutA', destination='Output1'),
                dict(source='Input1', destination='Slot3InA'),
                dict(source='Slot3OutA', destination='Slot4InA')]
m.set_connections(connections=connections)

m.set_frontend(1, impedance="50Ohm", attenuation="0dB", coupling="AC")

try:
    # Load CSV file
    lut_data = np.loadtxt('morse_AWG.csv', delimiter=',')

    # Convert to list
    lut_data_list = lut_data.tolist()

    # Use in AWG function
    awg.generate_waveform(channel=1, sample_rate='Auto', lut_data=lut_data_list, frequency=30e-3, amplitude=2)

    wg.generate_waveform(channel=1,type='Off')

    # Configure Lock In Amplifier settings
    lia.set_outputs(main="Theta", main_offset=0, aux='Demod', aux_offset=0)
    lia.set_polar_mode("25uVpp")
    lia.set_demodulation(mode="Internal",frequency=10e6,phase=0)
    lia.set_filter(corner_frequency=0.7,slope="Slope12dB")

    # Configure Digital Filter Box Settings
    dfb.set_control_matrix(1, input_gain1=1, input_gain2=0)
    dfb.set_filter(1, sample_rate='305.2kHz', shape='Lowpass', type='Bessel', low_corner=0.2, order=8, strict=False)
    dfb.set_monitor(1,'Output1') 

    # Configure WG Settings
    amp = 1000 # mVpp
    wg.generate_waveform(channel=1,type='Sine',frequency=10e6,amplitude=amp/1000)
    wg.set_modulation(channel=1,type='Phase',source='InputA',depth=90)

    # Sync instrument slots
    m.sync()

    # Attenuation setting based on external attenuators for filename
    attenuation = 100
    
    # Log output data
    response = dfb.start_logging(duration = 400, rate = 143) 
    FILE_PATH = "C:/Users/Your/Path/Here-" + str(attenuation) + "dB"
    file_name = response["file_name"]
    is_logging = True
    while is_logging:
        time.sleep(10)
        progress = dfb.logging_progress()
        remaining_time = int(progress['time_remaining'])
        print(f'Time remaining: {remaining_time} s')
        is_logging = remaining_time > 1

    temp_filename = FILE_PATH + "/" + str(attenuation) + "dB_" + str(amp) + "mVpp"
    dfb.download("ssd", file_name, temp_filename + ".li")
    os.system("mokucli convert --format=mat " + temp_filename + ".li")
    print("Downloaded log file to local directory.")

except Exception as e:
    print(f'Exception ocurred: {e}')
    traceback.print_exc()  # Prints the full traceback with line number
finally:
    print('reached relinquish ownership')
    m.relinquish_ownership()

Time remaining: 388 s
Time remaining: 379 s
Time remaining: 369 s
Time remaining: 359 s
Time remaining: 348 s
Time remaining: 339 s
Time remaining: 329 s
Time remaining: 319 s
Time remaining: 308 s
Time remaining: 299 s
Time remaining: 289 s
Time remaining: 279 s
Time remaining: 268 s
Time remaining: 259 s
Time remaining: 249 s
Time remaining: 238 s
Time remaining: 228 s
Time remaining: 219 s
Time remaining: 209 s
Time remaining: 198 s
Time remaining: 188 s
Time remaining: 179 s
Time remaining: 169 s
Time remaining: 158 s
Time remaining: 148 s
Time remaining: 138 s
Time remaining: 128 s
Time remaining: 118 s
Time remaining: 108 s
Time remaining: 98 s
Time remaining: 87 s
Time remaining: 77 s
Time remaining: 68 s
Time remaining: 58 s
Time remaining: 47 s
Time remaining: 37 s
Time remaining: 28 s
Time remaining: 18 s
Time remaining: 7 s
Time remaining: 0 s
Downloaded log file to local directory.
reached relinquish ownership
{'_content': b'<html>\r\n<head><title>502 Bad Gateway</title></h

MokuException: Can't connect the API server. Please check your Moku has the API server app installed by running `mokucli update`